In [1]:
"""this is same as repeat_representations, except we are going to try to do it with strategy 2: same Actor architecture, 
but manually repeat the weights in the actor_params object we are passing in"""

'this is same as repeat_representations, except we are going to try to do it with strategy 2: same Actor architecture, \nbut manually repeat the weights in the actor_params object we are passing in'

In [2]:
run_dir = "/scratch/gpfs/kw6487/JaxGCRL/clean_JaxGCRL/runs/humanoid_271_20250117-071637" #Humanoid, depth 8 (100k)
args_path = f"{run_dir}/args.pkl"
params_path = f"{run_dir}/final.pkl"

eval_env_id = None #if you want to use the env_id in args.eval_env_id, leave this as None

In [3]:
import pickle
import os
import jax
import flax
import tyro
import time
import optax
import wandb
import pickle
import random
import wandb_osh
import numpy as np
import flax.linen as nn
import jax.numpy as jnp
import matplotlib.pyplot as plt

from brax import envs
from etils import epath
from dataclasses import dataclass
from collections import namedtuple
from typing import NamedTuple, Any
from wandb_osh.hooks import TriggerWandbSyncHook
from flax.training.train_state import TrainState
from flax.linen.initializers import variance_scaling
from brax.io import html
from brax.io import model

from evaluator import CrlEvaluator
from buffer import TrajectoryUniformSamplingQueue
from memory_bank import MemoryBank, MemoryBankState

#COPY OVER DEFINITIONS
@dataclass
class Args:
    exp_name: str = "train" # os.path.basename(__file__)[: -len(".py")]
    seed: int = random.randint(1, 1000) # 16
    torch_deterministic: bool = True
    cuda: bool = True
    track: bool = True
    wandb_project_name: str = "clean_JaxGCRL_test"
    wandb_entity: str = 'wang-kevin3290-princeton-university'
    wandb_mode: str = 'offline'
    wandb_dir: str = '.'
    wandb_group: str = '.'
    capture_vis: bool = True
    vis_length: int = 1000
    checkpoint: bool = True

    #environment specific arguments
    env_id: str = "humanoid" # "ant_push" "ant_hardest_maze" "ant_big_maze" "humanoid" "ant"
    episode_length: int = 1000
    # to be filled in runtime
    obs_dim: int = 0
    goal_start_idx: int = 0
    goal_end_idx: int = 0

    # Algorithm specific arguments
    total_env_steps: int = 100000000 # 50000000
    num_epochs: int = 100 # 50
    num_envs: int = 512
    eval_env_id: str = ""
    num_eval_envs: int = 128
    actor_lr: float = 3e-4
    critic_lr: float = 3e-4
    alpha_lr: float = 3e-4
    batch_size: int = 256
    gamma: float = 0.99
    logsumexp_penalty_coeff: float = 0.1
    
    #adding in a batch_size_multiplier argument for critic vs. actor batch size
    critic_batch_size_multiplier: float = 1.0 #this has to be less than or equal to 1
    actor_batch_size_multiplier: float = 1.0 #this has to be less than 1

    max_replay_size: int = 10000
    min_replay_size: int = 1000
    
    unroll_length: int  = 62
    
    # ADDING IN A NETWORK WIDTH ARGUMENT
    same_network_width: int = 0
    network_width: int = 256
    critic_network_width: int = 256
    actor_network_width: int = 256
    actor_depth: int = 4
    critic_depth: int = 4
    actor_skip_connections: int = 0 # 0 for no skip connections, >= 0 means the frequency of skip connections (every X layers)
    critic_skip_connections: int = 0 # 0 for no skip connections, >= 0 means the frequency of skip connections (every X layers)
    
    num_episodes_per_env: int = 1 #the number of episodes to sample from each env when sampling data 
    #(to ensure number of batches is consistent as increase batch_size; for now, just a bandaid fix)
    # should be something like batch_size / 256
    training_steps_multiplier: int = 1 #should have the same effect as num_episodes_per_env, hmmm
    use_all_batches: int = 0 # if 1, use all batches; if 0, use a random subset of batches
    num_sgd_batches_per_training_step: int = 800 # this parameter so as to hold the number of batches constant (no matter batch_size, etc)
    
    mrn: int = 0
    memory_bank: int = 0
    memory_bank_size: int = batch_size # this can be modified too
    
    batchdiv2: int = 0 
    # if 1, freeze gradients for second half of batch
    # if 2, split in half along sa and freeze second half of g (Eysenbach ablation, remember it's forward loss)
    #
    # use batch_size * 2 and split in half and freeze gradients and all that (Eysenbach ablation), does not 
    # TODO: if 2, modifies actor such that it uses the half batch size (isolate for critic ablation)
    # can add 3, 4, etc (if diff between 1 and 2, maybe for batch_size ablation we need to have separate for actor and critic)
    # add more for instead of discarding second half, just freeze gradients for second half so symmetric with first
    
    eval_actor: int = 0
    # if 0, use deterministic actor for evaluation
    # if 1, use stochastic actor for evaluation
    # if 2, sample two actions and take the one with the higher Q value
    # if K >= 2, sample K actions and take the one with the highest Q value
    expl_actor: int = 1
    # if 0, use deterministic actor for exploration/collecting data
    # if 1, use stochastic actor for exploration/collecting data
    # if 2, sample two actions and take the one with the higher Q value
    # if K >= 2, sample K actions and take the one with the highest Q value
    
    entropy_param: float = 0.5
    disable_entropy: int = 0
    
    use_relu: int = 0
    
    resnet: str = "noishmistake4_nodense"
    
    num_render: int = 10
    
    
    
    # to be filled in runtime
    env_steps_per_actor_step : int = 0
    """number of env steps per actor step (computed in runtime)"""
    num_prefill_env_steps : int = 0
    """number of env steps to fill the buffer before starting training (computed in runtime)"""
    num_prefill_actor_steps : int = 0
    """number of actor steps to fill the buffer before starting training (computed in runtime)"""
    num_training_steps_per_epoch : int = 0
    """the number of training steps per epoch(computed in runtime)"""

def make_env(env_id, args):
    print(f"making env with env_id: {env_id}", flush=True)
    if env_id == "reacher":
        from envs.reacher import Reacher
        env = Reacher(
            backend="spring",
        )
        args.obs_dim = 10
        args.goal_start_idx = 4
        args.goal_end_idx = 7
    elif env_id == "pusher":
        from envs.pusher import Pusher
        env = Pusher(
            backend="spring",
        )
        args.obs_dim = 20
        args.goal_start_idx = 10
        args.goal_end_idx = 13
    elif env_id == "ant":
        from envs.ant import Ant
        env = Ant(
            backend="spring",
            exclude_current_positions_from_observation=False,
            terminate_when_unhealthy=True,
        )

        args.obs_dim = 29
        args.goal_start_idx = 0
        args.goal_end_idx = 2

    elif "ant" in env_id and "maze" in env_id: #needed the add the ant check to differentiate with humanoid maze
        if "gen" not in env_id:
            from envs.ant_maze import AntMaze
            env = AntMaze(
                backend="spring",
                exclude_current_positions_from_observation=False,
                terminate_when_unhealthy=True,
                maze_layout_name=env_id[4:]
            )

            # args.obs_dim = 29
            # args.goal_start_idx = 0
            # args.goal_end_idx = 2
        else:
            from envs.ant_maze_generalization import AntMazeGeneralization
            gen_idx = env_id.find("gen")
            maze_layout_name = env_id[4:gen_idx-1]
            generalization_config = env_id[gen_idx+4:]
            print(f"maze_layout_name: {maze_layout_name}, generalization_config: {generalization_config}", flush=True)
            env = AntMazeGeneralization(
                backend="spring",
                exclude_current_positions_from_observation=False,
                terminate_when_unhealthy=True,
                maze_layout_name=maze_layout_name,
                generalization_config=generalization_config
            )

            args.obs_dim = 29
            args.goal_start_idx = 0
            args.goal_end_idx = 2
    
    elif env_id == "ant_ball":
        from envs.ant_ball import AntBall
        env = AntBall(
            backend="spring",
            exclude_current_positions_from_observation=False,
            terminate_when_unhealthy=True,
        )

        args.obs_dim = 31
        args.goal_start_idx = 28
        args.goal_end_idx = 30

    elif env_id == "ant_push":
        from envs.ant_push import AntPush
        env = AntPush(
            backend="mjx",
        )

        args.obs_dim = 31
        args.goal_start_idx = 0
        args.goal_end_idx = 2
        
    elif env_id == "humanoid":
        from envs.humanoid import Humanoid
        env = Humanoid(
            backend="spring",
            exclude_current_positions_from_observation=False,
            terminate_when_unhealthy=True,
        )

        args.obs_dim = 268
        args.goal_start_idx = 0
        args.goal_end_idx = 3
        
    elif "humanoid" in env_id and "maze" in env_id:
        from envs.humanoid_maze import HumanoidMaze
        env = HumanoidMaze(
            backend="spring",
            maze_layout_name=env_id[9:]
        )

        args.obs_dim = 268
        args.goal_start_idx = 0
        args.goal_end_idx = 3

        
    elif env_id == "arm_reach":
        from envs.manipulation.arm_reach import ArmReach
        env = ArmReach(
            backend="mjx",
        )

        args.obs_dim = 13
        args.goal_start_idx = 7
        args.goal_end_idx = 10
        
    elif env_id == "arm_binpick_easy":
        from envs.manipulation.arm_binpick_easy import ArmBinpickEasy
        env = ArmBinpickEasy(
            backend="mjx",
        )

        args.obs_dim = 17
        args.goal_start_idx = 0
        args.goal_end_idx = 3
        
    elif env_id == "arm_binpick_hard":
        from envs.manipulation.arm_binpick_hard import ArmBinpickHard
        env = ArmBinpickHard(
            backend="mjx",
        )

        args.obs_dim = 17
        args.goal_start_idx = 0
        args.goal_end_idx = 3
        
    elif env_id == "arm_binpick_easy_EEF":
        from envs.manipulation.arm_binpick_easy_EEF import ArmBinpickEasyEEF
        env = ArmBinpickEasyEEF(
            backend="mjx",
        )

        args.obs_dim = 11
        args.goal_start_idx = 0
        args.goal_end_idx = 3
    
    elif "arm_grasp" in env_id: # either arm_grasp or arm_grasp_0.5, etc
        from envs.manipulation.arm_grasp import ArmGrasp
        cube_noise_scale = float(env_id[10:]) if len(env_id) > 9 else 0.3
        env = ArmGrasp(
            cube_noise_scale=cube_noise_scale,
            backend="mjx",
        )

        args.obs_dim = 23
        args.goal_start_idx = 16
        args.goal_end_idx = 23
    
    elif env_id == "arm_push_easy":
        from envs.manipulation.arm_push_easy import ArmPushEasy
        env = ArmPushEasy(
            backend="mjx",
        )

        args.obs_dim = 17
        args.goal_start_idx = 0
        args.goal_end_idx = 3
    
    elif env_id == "arm_push_hard":
        from envs.manipulation.arm_push_hard import ArmPushHard
        env = ArmPushHard(
            backend="mjx",
        )

        args.obs_dim = 17
        args.goal_start_idx = 0
        args.goal_end_idx = 3

    else:
        raise NotImplementedError
    
    return env

lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
bias_init = nn.initializers.zeros
def residual_block(x, width, normalize, activation):
    identity = x
    x = nn.Dense(width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
    x = normalize(x)
    x = activation(x)
    x = nn.Dense(width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
    x = normalize(x)
    x = activation(x)
    x = nn.Dense(width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
    x = normalize(x)
    x = activation(x)
    x = nn.Dense(width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
    x = normalize(x)
    x = activation(x)
    x = x + identity
    return x

class SA_encoder(nn.Module):
    norm_type = "layer_norm"
    network_width: int = 1024
    network_depth: int = 4
    skip_connections: int = 0
    use_relu: int = 0
    @nn.compact
    def __call__(self, s: jnp.ndarray, a: jnp.ndarray):

        lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros
        
        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x
        
        if self.use_relu:
            activation = nn.relu
        else:
            activation = nn.swish
            
        x = jnp.concatenate([s, a], axis=-1)
        #Initial layer
        x = nn.Dense(self.network_width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = activation(x)
        #Residual blocks
        for i in range(self.network_depth // 4):
            x = residual_block(x, self.network_width, normalize, activation)
        #Final layer
        x = nn.Dense(64, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        return x
    
class G_encoder(nn.Module):
    norm_type = "layer_norm"
    network_width: int = 1024
    network_depth: int = 4
    skip_connections: int = 0
    use_relu: int = 0
    @nn.compact
    def __call__(self, g: jnp.ndarray):

        lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros

        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x
        
        if self.use_relu:
            activation = nn.relu
        else:
            activation = nn.swish
        
        x = g
        #Initial layer
        x = nn.Dense(self.network_width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = activation(x)
        #Residual blocks
        for i in range(self.network_depth // 4):
            x = residual_block(x, self.network_width, normalize, activation)
        #Final layer
        x = nn.Dense(64, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        return x

class Actor(nn.Module):
    action_size: int
    norm_type = "layer_norm"
    network_width: int = 1024
    network_depth: int = 4
    skip_connections: int = 0 # 0 for no skip connections, >= 0 means the frequency of skip connections (every X layers)
    use_relu: int = 0
    LOG_STD_MAX = 2
    LOG_STD_MIN = -5

    @nn.compact
    def __call__(self, x):
        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x
            
        if self.use_relu:
            activation = nn.relu
        else:
            activation = nn.swish

        lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros
        
        #Initial layer
        x = nn.Dense(self.network_width, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = activation(x)
        #Residual blocks
        for i in range(self.network_depth // 4):
            x = residual_block(x, self.network_width, normalize, activation)
        #Final layer
        # x = nn.Dense(64, kernel_init=lecun_unfirom, bias_init=bias_init)(x)

        mean = nn.Dense(self.action_size, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        log_std = nn.Dense(self.action_size, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        
        log_std = nn.tanh(log_std)
        log_std = self.LOG_STD_MIN + 0.5 * (self.LOG_STD_MAX - self.LOG_STD_MIN) * (log_std + 1)  # From SpinUp / Denis Yarats

        return mean, log_std

In [4]:
import pickle
with open(args_path, 'rb') as f:
    args = pickle.load(f)

In [5]:
# Create random key
key = jax.random.PRNGKey(args.seed)
key, env_key, eval_env_key = jax.random.split(key, 3)

# Create eval environment
if args.eval_env_id:
    eval_env = make_env(args.eval_env_id, args)
else:
    eval_env = make_env()
eval_env = envs.training.wrap(
    eval_env,
    episode_length=args.episode_length,
)

obs_size = eval_env.observation_size
action_size = eval_env.action_size

making env with env_id: humanoid


In [6]:
params = model.load_params(params_path)
alpha_params, actor_params, critic_params = params
sa_encoder_params, g_encoder_params = critic_params['sa_encoder'], critic_params['g_encoder']
actor = Actor(action_size=action_size, network_width=args.actor_network_width, network_depth=args.actor_depth, skip_connections=args.actor_skip_connections, use_relu=args.use_relu)
sa_encoder = SA_encoder(network_width=args.critic_network_width, network_depth=args.critic_depth, skip_connections=args.critic_skip_connections, use_relu=args.use_relu)
g_encoder = G_encoder(network_width=args.critic_network_width, network_depth=args.critic_depth, skip_connections=args.critic_skip_connections, use_relu=args.use_relu)

In [7]:
actor_params['params'].keys()

dict_keys(['Dense_0', 'Dense_1', 'Dense_10', 'Dense_2', 'Dense_3', 'Dense_4', 'Dense_5', 'Dense_6', 'Dense_7', 'Dense_8', 'Dense_9', 'LayerNorm_0', 'LayerNorm_1', 'LayerNorm_2', 'LayerNorm_3', 'LayerNorm_4', 'LayerNorm_5', 'LayerNorm_6', 'LayerNorm_7', 'LayerNorm_8'])

In [9]:
NUM_REPEAT_LAYERS = 4 #this is added layers
NUM_REPEAT_RESIDUAL_BLOCKS = NUM_REPEAT_LAYERS // args.actor_skip_connections + 1 #this means a total of 2, not 2 repeat (i.e. 1 means no repeats, recover original)
NEW_DEPTH = args.actor_depth + (NUM_REPEAT_RESIDUAL_BLOCKS - 1) * args.actor_skip_connections
print(f"NUM_REPEAT_RESIDUAL_BLOCKS: {NUM_REPEAT_RESIDUAL_BLOCKS}")
print(f"OLD_DEPTH: {args.actor_depth}, NEW_DEPTH: {NEW_DEPTH}")

NUM_REPEAT_RESIDUAL_BLOCKS: 2
OLD_DEPTH: 8, NEW_DEPTH: 12


In [10]:
actor2 = Actor(action_size=action_size, network_width=args.actor_network_width, network_depth=NEW_DEPTH, skip_connections=args.actor_skip_connections, use_relu=args.use_relu)
key, actor2_key = jax.random.split(key, 2)
actor2_params = actor2.init(actor2_key, np.ones([1, obs_size]))

In [11]:
# dense_mappings = {} #maps actor_params -> actor2_params
# N_actor, N_actor2 = args.actor_depth + 3, NEW_DEPTH + 3 #the num of Dense layers for each i.e 10, 14 (in our example case)
# assert len([x for x in actor_params['params'].keys() if 'Dense' in x]) == N_actor
# assert len([x for x in actor2_params['params'].keys() if 'Dense' in x]) == N_actor2
# print(N_actor, N_actor2)

In [12]:
dense_mappings = {} #maps actor2_params idx -> actor_params idx
N_actor, N_actor2 = args.actor_depth + 3, NEW_DEPTH + 3 #the num of Dense layers for each i.e 11, 15 (in our example case)
assert len([x for x in actor_params['params'].keys() if 'Dense' in x]) == N_actor
assert len([x for x in actor2_params['params'].keys() if 'Dense' in x]) == N_actor2
print(N_actor, N_actor2)

copy_range = np.arange(N_actor-2-args.actor_skip_connections, N_actor-2) #INCLUSIVE, the range of the layers in actor that are being copied

#okay so let's just copy over everything to start (based on mod), then we'll fix
for i in range(N_actor2):
    idx = (i-1) % 4
    dense_mappings[i] = copy_range[idx]

#now we fix
dense_mappings[0] = 0
for i in range(1, args.actor_depth+1):
    dense_mappings[i] = i
dense_mappings[N_actor2-2] = N_actor-2
dense_mappings[N_actor2-1] = N_actor-1

# for k, v in dense_mappings.items():
#     print(f"Dense_{k} <- Dense_{v}")

for k, v in dense_mappings.items():
    str_k, str_v = f'Dense_{k}', f'Dense_{v}'
    actor2_params['params'][str_k] = actor_params['params'][str_v]
    print(f"Setting actor2_params's {str_k} with actor_params's {str_v}")

11 15
Setting actor2_params's Dense_0 with actor_params's Dense_0
Setting actor2_params's Dense_1 with actor_params's Dense_1
Setting actor2_params's Dense_2 with actor_params's Dense_2
Setting actor2_params's Dense_3 with actor_params's Dense_3
Setting actor2_params's Dense_4 with actor_params's Dense_4
Setting actor2_params's Dense_5 with actor_params's Dense_5
Setting actor2_params's Dense_6 with actor_params's Dense_6
Setting actor2_params's Dense_7 with actor_params's Dense_7
Setting actor2_params's Dense_8 with actor_params's Dense_8
Setting actor2_params's Dense_9 with actor_params's Dense_5
Setting actor2_params's Dense_10 with actor_params's Dense_6
Setting actor2_params's Dense_11 with actor_params's Dense_7
Setting actor2_params's Dense_12 with actor_params's Dense_8
Setting actor2_params's Dense_13 with actor_params's Dense_9
Setting actor2_params's Dense_14 with actor_params's Dense_10


In [13]:
ln_mappings = {} #maps actor2_params idx -> actor_params idx
LN_actor, LN_actor2 = args.actor_depth + 1, NEW_DEPTH + 1 
assert len([x for x in actor_params['params'].keys() if 'LayerNorm' in x]) == LN_actor
assert len([x for x in actor2_params['params'].keys() if 'LayerNorm' in x]) == LN_actor2
print(LN_actor, LN_actor2)

copy_range = np.arange(LN_actor-args.actor_skip_connections, LN_actor) #should be equivalent


#okay so let's just copy over everything to start (based on mod), then we'll fix
for i in range(LN_actor2):
    idx = (i-1) % 4
    ln_mappings[i] = copy_range[idx]

#now we fix
ln_mappings[0] = 0
for i in range(1, args.actor_depth+1):
    ln_mappings[i] = i

# for k, v in ln_mappings.items():
#     print(f"Dense_{k} <- Dense_{v}")

for k, v in ln_mappings.items():
    str_k, str_v = f'LayerNorm_{k}', f'LayerNorm_{v}'
    actor2_params['params'][str_k] = actor_params['params'][str_v]
    print(f"Setting actor2_params's {str_k} with actor_params's {str_v}")

9 13
Setting actor2_params's LayerNorm_0 with actor_params's LayerNorm_0
Setting actor2_params's LayerNorm_1 with actor_params's LayerNorm_1
Setting actor2_params's LayerNorm_2 with actor_params's LayerNorm_2
Setting actor2_params's LayerNorm_3 with actor_params's LayerNorm_3
Setting actor2_params's LayerNorm_4 with actor_params's LayerNorm_4
Setting actor2_params's LayerNorm_5 with actor_params's LayerNorm_5
Setting actor2_params's LayerNorm_6 with actor_params's LayerNorm_6
Setting actor2_params's LayerNorm_7 with actor_params's LayerNorm_7
Setting actor2_params's LayerNorm_8 with actor_params's LayerNorm_8
Setting actor2_params's LayerNorm_9 with actor_params's LayerNorm_5
Setting actor2_params's LayerNorm_10 with actor_params's LayerNorm_6
Setting actor2_params's LayerNorm_11 with actor_params's LayerNorm_7
Setting actor2_params's LayerNorm_12 with actor_params's LayerNorm_8


In [14]:
actor2_params['params'].keys()

dict_keys(['Dense_0', 'LayerNorm_0', 'Dense_1', 'LayerNorm_1', 'Dense_2', 'LayerNorm_2', 'Dense_3', 'LayerNorm_3', 'Dense_4', 'LayerNorm_4', 'Dense_5', 'LayerNorm_5', 'Dense_6', 'LayerNorm_6', 'Dense_7', 'LayerNorm_7', 'Dense_8', 'LayerNorm_8', 'Dense_9', 'LayerNorm_9', 'Dense_10', 'LayerNorm_10', 'Dense_11', 'LayerNorm_11', 'Dense_12', 'LayerNorm_12', 'Dense_13', 'Dense_14'])

In [15]:
assert actor2_params['params']['Dense_10']['kernel'][12][-4] == actor_params['params']['Dense_6']['kernel'][12][-4]
assert actor2_params['params']['LayerNorm_11']['scale'][12] == actor_params['params']['LayerNorm_7']['scale'][12]

In [16]:
dummy_input = jnp.ones([1, obs_size])

key, inference_key = jax.random.split(key, 2)
means, log_stds = actor2.apply(actor2_params, dummy_input)

# # Calculate standard deviations from log standard deviations
# stds = jnp.exp(log_stds)

# # Sample an action (as would be done during policy execution)
# action_sampled = nn.tanh(means + stds * jax.random.normal(inference_key, shape=means.shape, dtype=means.dtype))

# # Print shape and values
# print(f"Input shape: {dummy_input.shape}")
# print(f"Mean action shape: {means.shape}")
# print(f"Log std shape: {log_stds.shape}")
# print(f"Sampled action shape: {action_sampled.shape}")
# print(f"Mean action values: {means[0][:5]}")  # First 5 values of the first example
# print(f"Sampled action values: {action_sampled[0][:5]}")  # First 5 values of the first example

In [17]:
means

Array([[ 2.2558    , -4.6055655 , -1.8280262 ,  1.902463  ,  1.9668996 ,
        -1.2866569 ,  0.53597414, -1.4512815 ,  2.7577653 , -1.088382  ,
         5.670095  ,  1.7182587 , -0.1120318 , -1.5639436 ,  1.3610101 ,
         1.8614221 , -2.2567968 ]], dtype=float32)

In [18]:
dummy_input

Array([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 

In [8]:
### NOW WE WORK ON ACTOR 3
NUM_REPEAT_LAYERS = args.actor_skip_connections * 1 #this is added layers
NEW_DEPTH = args.actor_depth + NUM_REPEAT_LAYERS
print(f"NUM_REPEAT_LAYERS: {NUM_REPEAT_LAYERS}")
print(f"OLD_DEPTH: {args.actor_depth}, NEW_DEPTH: {NEW_DEPTH}")

NUM_REPEAT_LAYERS: 4
OLD_DEPTH: 8, NEW_DEPTH: 12


In [10]:
actor3 = Actor(action_size=action_size, network_width=args.actor_network_width, network_depth=NEW_DEPTH, skip_connections=args.actor_skip_connections, use_relu=args.use_relu)
key, actor3_key = jax.random.split(key, 2)
actor3_params = actor3.init(actor3_key, np.ones([1, obs_size]))

In [11]:
dense_mappings = {} #maps actor3_params idx -> actor_params idx
N_actor, N_actor3 = args.actor_depth + 3, NEW_DEPTH + 3 #the num of Dense layers for each i.e 11, 15 (in our example case)
assert len([x for x in actor_params['params'].keys() if 'Dense' in x]) == N_actor
assert len([x for x in actor3_params['params'].keys() if 'Dense' in x]) == N_actor3
print(N_actor, N_actor3)

# copy_range = np.arange(N_actor-2-args.actor_skip_connections, N_actor-2) #INCLUSIVE, the range of the layers in actor that are being copied
copy_idx = args.actor_depth

#okay so let's just copy over everything to start (based on mod), then we'll fix
for i in range(N_actor3):
    dense_mappings[i] = copy_idx

#now we fix
dense_mappings[0] = 0
for i in range(1, args.actor_depth+1):
    dense_mappings[i] = i
dense_mappings[N_actor3-2] = N_actor-2
dense_mappings[N_actor3-1] = N_actor-1

# for k, v in dense_mappings.items():
#     print(f"Dense_{k} <- Dense_{v}")

for k, v in dense_mappings.items():
    str_k, str_v = f'Dense_{k}', f'Dense_{v}'
    actor3_params['params'][str_k] = actor_params['params'][str_v]
    print(f"Setting actor3_params's {str_k} with actor_params's {str_v}")

11 15
Setting actor3_params's Dense_0 with actor_params's Dense_0
Setting actor3_params's Dense_1 with actor_params's Dense_1
Setting actor3_params's Dense_2 with actor_params's Dense_2
Setting actor3_params's Dense_3 with actor_params's Dense_3
Setting actor3_params's Dense_4 with actor_params's Dense_4
Setting actor3_params's Dense_5 with actor_params's Dense_5
Setting actor3_params's Dense_6 with actor_params's Dense_6
Setting actor3_params's Dense_7 with actor_params's Dense_7
Setting actor3_params's Dense_8 with actor_params's Dense_8
Setting actor3_params's Dense_9 with actor_params's Dense_8
Setting actor3_params's Dense_10 with actor_params's Dense_8
Setting actor3_params's Dense_11 with actor_params's Dense_8
Setting actor3_params's Dense_12 with actor_params's Dense_8
Setting actor3_params's Dense_13 with actor_params's Dense_9
Setting actor3_params's Dense_14 with actor_params's Dense_10


In [13]:
ln_mappings = {} #maps actor3_params idx -> actor_params idx
LN_actor, LN_actor3 = args.actor_depth + 1, NEW_DEPTH + 1 
assert len([x for x in actor_params['params'].keys() if 'LayerNorm' in x]) == LN_actor
assert len([x for x in actor3_params['params'].keys() if 'LayerNorm' in x]) == LN_actor3
print(LN_actor, LN_actor3)

# copy_range = np.arange(LN_actor-args.actor_skip_connections, LN_actor) #should be equivalent
copy_idx = args.actor_depth

#okay so let's just copy over everything to start (based on mod), then we'll fix
for i in range(LN_actor3):
    ln_mappings[i] = copy_idx

#now we fix
ln_mappings[0] = 0
for i in range(1, args.actor_depth+1):
    ln_mappings[i] = i

# for k, v in ln_mappings.items():
#     print(f"Dense_{k} <- Dense_{v}")

for k, v in ln_mappings.items():
    str_k, str_v = f'LayerNorm_{k}', f'LayerNorm_{v}'
    actor3_params['params'][str_k] = actor_params['params'][str_v]
    print(f"Setting actor3_params's {str_k} with actor_params's {str_v}")

9 13
Setting actor3_params's LayerNorm_0 with actor_params's LayerNorm_0
Setting actor3_params's LayerNorm_1 with actor_params's LayerNorm_1
Setting actor3_params's LayerNorm_2 with actor_params's LayerNorm_2
Setting actor3_params's LayerNorm_3 with actor_params's LayerNorm_3
Setting actor3_params's LayerNorm_4 with actor_params's LayerNorm_4
Setting actor3_params's LayerNorm_5 with actor_params's LayerNorm_5
Setting actor3_params's LayerNorm_6 with actor_params's LayerNorm_6
Setting actor3_params's LayerNorm_7 with actor_params's LayerNorm_7
Setting actor3_params's LayerNorm_8 with actor_params's LayerNorm_8
Setting actor3_params's LayerNorm_9 with actor_params's LayerNorm_8
Setting actor3_params's LayerNorm_10 with actor_params's LayerNorm_8
Setting actor3_params's LayerNorm_11 with actor_params's LayerNorm_8
Setting actor3_params's LayerNorm_12 with actor_params's LayerNorm_8


In [19]:
assert actor3_params['params']['Dense_10']['kernel'][12][-4] == actor_params['params']['Dense_8']['kernel'][12][-4]
assert actor3_params['params']['Dense_7']['kernel'][12][-4] == actor_params['params']['Dense_7']['kernel'][12][-4]
assert actor3_params['params']['LayerNorm_11']['scale'][12] == actor_params['params']['LayerNorm_8']['scale'][12]
assert actor3_params['params']['LayerNorm_4']['scale'][12] == actor_params['params']['LayerNorm_4']['scale'][12]

In [20]:
actor3_params['params'].keys()

dict_keys(['Dense_0', 'LayerNorm_0', 'Dense_1', 'LayerNorm_1', 'Dense_2', 'LayerNorm_2', 'Dense_3', 'LayerNorm_3', 'Dense_4', 'LayerNorm_4', 'Dense_5', 'LayerNorm_5', 'Dense_6', 'LayerNorm_6', 'Dense_7', 'LayerNorm_7', 'Dense_8', 'LayerNorm_8', 'Dense_9', 'LayerNorm_9', 'Dense_10', 'LayerNorm_10', 'Dense_11', 'LayerNorm_11', 'Dense_12', 'LayerNorm_12', 'Dense_13', 'Dense_14'])

In [21]:
dummy_input = jnp.ones([1, obs_size])

key, inference_key = jax.random.split(key, 2)
means, log_stds = actor3.apply(actor3_params, dummy_input)

# # Calculate standard deviations from log standard deviations
# stds = jnp.exp(log_stds)

# # Sample an action (as would be done during policy execution)
# action_sampled = nn.tanh(means + stds * jax.random.normal(inference_key, shape=means.shape, dtype=means.dtype))

# # Print shape and values
# print(f"Input shape: {dummy_input.shape}")
# print(f"Mean action shape: {means.shape}")
# print(f"Log std shape: {log_stds.shape}")
# print(f"Sampled action shape: {action_sampled.shape}")
# print(f"Mean action values: {means[0][:5]}")  # First 5 values of the first example
# print(f"Sampled action values: {action_sampled[0][:5]}")  # First 5 values of the first example
means

Array([[ 4.1045327 , -5.5106215 , -1.5605875 ,  0.53137565, -0.7361961 ,
         0.40704456,  0.86970675, -1.3488284 ,  1.7122121 , -1.5652541 ,
         5.5913796 ,  1.4178748 ,  0.10758056, -0.3807908 ,  2.268794  ,
         0.96278703, -0.91206694]], dtype=float32)